In [1]:
import os
import json
import cv2
import shutil
import random
import copy
from pathlib import Path
from collections import defaultdict
import albumentations as A
from tqdm import tqdm

# =====================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# =====================================================================
INPUT_DATASETS = [
    {
        "prefix": "gold",
        "img_dir": Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/images"),
        "meta_path": Path("/kaggle/input/datasets/quii29/rukopys-dataset/train/metadata.jsonl")
    },
    {
        "prefix": "silver",
        "img_dir": Path("/kaggle/input/datasets/quii29/rukopys-silver-rare-classes/images"),
        "meta_path": Path("/kaggle/input/datasets/quii29/rukopys-silver-rare-classes/metadata.jsonl")
    }
]

OUT_ROOT = Path("/kaggle/working/Merged_Rukopys_V1")
OUT_TRAIN = OUT_ROOT / "train"
OUT_IMG_DIR = OUT_TRAIN / "images"
OUT_META_PATH = OUT_TRAIN / "metadata.jsonl"
OUT_COCO_PATH = OUT_TRAIN / "annotations_detection.json"
OUT_REL_PATH = OUT_TRAIN / "annotations_relations.json"

DEBUG_DIR = Path("/kaggle/working/debug_merged")
ZIP_OUT_PATH = "/kaggle/working/Merged_Rukopys_V1" 

BBOX_FORMAT = "coco" 
MAX_REPEATS = 4
MAX_IMAGE_SIZE = 4000 # Giới hạn size để tránh OOM / Decompression Bomb

HEAD_CLASSES = {'handwritten', 'formula'}
MID_CLASSES = {'printed', 'annotation', 'table'}
TAIL_CLASSES = {'image', 'graph'}

READABLE_CLASSES = {'handwritten', 'printed', 'annotation', 'formula'}
CLASS_NAME_TO_ID = {
    'handwritten': 1, 'printed': 2, 'formula': 3, 
    'table': 4, 'annotation': 5, 'image': 6, 'graph': 7
}

# =====================================================================
# 2. HÀM CHUYỂN ĐỔI, KIỂM TRA BBOX & PHASE 3 LOGIC
# =====================================================================
def original_bbox_to_coco(bbox, fmt):
    if fmt == "coco": return bbox
    if fmt == "xyxy": return [bbox[0], bbox[1], bbox[2] - bbox[0], bbox[3] - bbox[1]]
    raise ValueError(f"Unsupported format: {fmt}")

def coco_bbox_to_original(bbox, fmt):
    if fmt == "coco": return bbox
    if fmt == "xyxy": return [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
    raise ValueError(f"Unsupported format: {fmt}")

def clamp_coco_bbox(bbox, img_w, img_h):
    x, y, w, h = bbox
    # FIX: Ép x, y nằm cứng bên trong giới hạn ảnh (trừ hao 2 pixel)
    x = max(0.0, min(float(x), float(img_w) - 2.0))
    y = max(0.0, min(float(y), float(img_h) - 2.0))
    # Đảm bảo w, h luôn > 1 và không tràn qua viền phải/dưới
    w = max(1.0, min(float(w), float(img_w) - x))
    h = max(1.0, min(float(h), float(img_h) - y))
    return [x, y, w, h]

def generate_reading_order_edges(regions):
    readable_items = []
    for idx, r in enumerate(regions):
        c_type = r.get('type', '')
        if c_type in READABLE_CLASSES and 'bbox' in r:
            x, y, w, h = r['bbox']
            if BBOX_FORMAT != "coco":
                x, y, w, h = original_bbox_to_coco([x,y,w,h], BBOX_FORMAT)
            cx = x + w / 2.0
            cy = y + h / 2.0
            readable_items.append({'idx': idx, 'cx': cx, 'cy': cy, 'h': h})
            
    if not readable_items: return []
        
    readable_items.sort(key=lambda item: item['cy'])
    
    lines = []
    current_line = [readable_items[0]]
    for item in readable_items[1:]:
        prev_item = current_line[-1]
        threshold = max(prev_item['h'], item['h']) * 0.5 
        if abs(item['cy'] - prev_item['cy']) < threshold:
            current_line.append(item)
        else:
            lines.append(current_line)
            current_line = [item]
    if current_line: lines.append(current_line)
        
    ordered_indices = []
    for line in lines:
        line.sort(key=lambda item: item['cx'])
        ordered_indices.extend([item['idx'] for item in line])
        
    edges = []
    for i in range(len(ordered_indices) - 1):
        edges.append([ordered_indices[i], ordered_indices[i+1], "next"])
        
    return edges

# =====================================================================
# 3. LOGIC TÁI CÂN BẰNG
# =====================================================================
def compute_image_priority(regions):
    counts = defaultdict(int)
    for r in regions: counts[r.get('type', 'unknown')] += 1
        
    total_boxes = sum(counts.values())
    if total_boxes == 0: return {"repeats": 0, "pipeline": "aug_light", "stats": counts}

    head_boxes = sum(counts[c] for c in HEAD_CLASSES)
    mid_boxes = sum(counts[c] for c in MID_CLASSES)
    tail_boxes = sum(counts[c] for c in TAIL_CLASSES)
    
    rare_ratio = tail_boxes / total_boxes
    dominant_ratio = head_boxes / total_boxes

    if dominant_ratio >= 0.8 and tail_boxes <= 1:
        return {"repeats": 0, "pipeline": "aug_light", "stats": counts}

    score = (tail_boxes * 3.0 + mid_boxes * 1.2 + rare_ratio * 8.0 + counts['image'] * 2.0 + counts['graph'] * 3.0) - (dominant_ratio * 6.0 + max(0, counts['handwritten'] - 20) * 0.15 + max(0, counts['formula'] - 10) * 0.20)

    repeats = 0
    pipeline = "aug_light"

    if tail_boxes > 0 and rare_ratio >= 0.1:
        repeats = random.choice([3, 4])
        pipeline = "aug_rare"
    elif tail_boxes > 0 or mid_boxes > 5:
        repeats = random.choice([1, 2])
        pipeline = "aug_rare" if tail_boxes > 0 else "aug_dense"
    elif counts['handwritten'] >= 15:
        repeats = 1
        pipeline = "aug_dense"
    elif score > 0:
        repeats = random.choice([0, 1])
        pipeline = "aug_light"

    return {"repeats": min(repeats, MAX_REPEATS), "pipeline": pipeline, "stats": counts}

# =====================================================================
# 4. KHỞI TẠO AUGMENTATION PIPELINES
# =====================================================================
bbox_params = A.BboxParams(format='coco', label_fields=['region_idx'], min_visibility=0.5)

pipelines = {
    "aug_light": A.Compose([A.RandomBrightnessContrast(p=0.5), A.GaussNoise(p=0.3), A.Blur(blur_limit=3, p=0.2)], bbox_params=bbox_params),
    "aug_dense": A.Compose([A.Affine(rotate=(-3, 3), translate_percent={"x": (-0.02, 0.02), "y": (-0.02, 0.02)}, p=0.7), A.RandomBrightnessContrast(p=0.5), A.ISONoise(p=0.3)], bbox_params=bbox_params),
    "aug_rare": A.Compose([A.Affine(rotate=(-3, 3), scale=(0.97, 1.03), p=0.8), A.RandomBrightnessContrast(p=0.6)], bbox_params=bbox_params)
}

# =====================================================================
# 5. DỌN DẸP
# =====================================================================
def cleanup_working_except_zip(zip_path):
    print("🧹 Đang dọn dẹp toàn bộ dữ liệu trung gian trong /kaggle/working...")
    for item in Path("/kaggle/working").iterdir():
        if item.name == Path(zip_path).name: continue
        if item.is_dir(): shutil.rmtree(item)
        else: item.unlink()
    print("✅ Cleanup hoàn tất. Chỉ giữ lại file Zip.")

# =====================================================================
# 6. CHƯƠNG TRÌNH CHÍNH
# =====================================================================
def main():
    if OUT_ROOT.exists(): shutil.rmtree(OUT_ROOT)
    if DEBUG_DIR.exists(): shutil.rmtree(DEBUG_DIR)
    
    OUT_IMG_DIR.mkdir(parents=True, exist_ok=True)
    DEBUG_DIR.mkdir(parents=True, exist_ok=True)
    
    stats = {
        "original_images": 0, "augmented_images": 0, "failed_augments": 0, "invalid_boxes_dropped": 0,
        "boxes_before": defaultdict(int), "boxes_added": defaultdict(int)
    }
    
    coco_dataset = {"images": [], "annotations": [], "categories": [{"id": v, "name": k} for k, v in CLASS_NAME_TO_ID.items()]}
    relations_dataset = []
    
    global_img_id = 1
    global_ann_id = 1
    debug_samples = []
    
    out_meta_file = open(OUT_META_PATH, 'w', encoding='utf-8')
    
    def process_and_add_to_datasets(img_name, img_w, img_h, regions_data):
        nonlocal global_img_id, global_ann_id
        coco_dataset["images"].append({"id": global_img_id, "file_name": img_name, "width": img_w, "height": img_h})
        
        rel_record = {"image_id": global_img_id, "file_name": img_name, "regions": [], "edges": []}
        
        for r_idx, r in enumerate(regions_data):
            c_type = r.get('type')
            cat_id = CLASS_NAME_TO_ID.get(c_type)
            rel_record["regions"].append({"region_idx": r_idx, "type": c_type, "bbox": r.get('bbox')})
            
            if cat_id is not None and 'bbox' in r:
                coco_box = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                _, _, w, h = coco_box
                coco_dataset["annotations"].append({
                    "id": global_ann_id, "image_id": global_img_id, "category_id": cat_id,
                    "bbox": [round(float(v), 2) for v in coco_box], "area": round(float(w * h), 2),
                    "iscrowd": 0, "region_idx_in_image": r_idx
                })
                global_ann_id += 1
                
        rel_record["edges"] = generate_reading_order_edges(regions_data)
        relations_dataset.append(rel_record)
        global_img_id += 1

    print("🚀 Bắt đầu gộp và xử lý bộ dữ liệu Merged_Rukopys_V1...")
    
    for ds_info in INPUT_DATASETS:
        prefix = ds_info["prefix"]
        in_img_dir = ds_info["img_dir"]
        in_meta_path = ds_info["meta_path"]
        
        if not in_meta_path.exists():
            print(f"⚠️ Bỏ qua {prefix} do không tìm thấy metadata tại: {in_meta_path}")
            continue
            
        print(f"\n📂 Đang xử lý tập dữ liệu: [{prefix.upper()}]")
        with open(in_meta_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        for line in tqdm(lines, desc=f"Processing {prefix}"):
            record = json.loads(line)
            fname_rel = record.get('file_name', '')
            orig_fname = Path(fname_rel).name
            safe_fname = f"{prefix}_{orig_fname}"
            in_img_path = in_img_dir / orig_fname
            
            if not in_img_path.exists(): continue
                
            # -- FIX 1 & 2: Đọc bằng cv2 để reset EXIF/Permission + Resize chặn OOM --
            img = cv2.imread(str(in_img_path))
            if img is None: 
                continue # Bỏ qua luôn ảnh hỏng gốc
                
            img_h, img_w = img.shape[:2]
            regions = record.get('regions', [])
            scale = 1.0
            
            # Khống chế ảnh khổng lồ (vượt MAX_IMAGE_SIZE)
            if img_w > MAX_IMAGE_SIZE or img_h > MAX_IMAGE_SIZE:
                scale = MAX_IMAGE_SIZE / max(img_w, img_h)
                new_w, new_h = int(img_w * scale), int(img_h * scale)
                img = cv2.resize(img, (new_w, new_h))
                img_w, img_h = new_w, new_h
                
                # Cập nhật scale vào tọa độ boxes
                for r in regions:
                    if 'bbox' in r and len(r['bbox']) == 4:
                        box = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                        box = [box[0]*scale, box[1]*scale, box[2]*scale, box[3]*scale]
                        r['bbox'] = coco_bbox_to_original(box, BBOX_FORMAT)

            for r in regions: stats["boxes_before"][r.get('type', 'unknown')] += 1
                
            # Cập nhật metadata record theo chuẩn mới
            record['image_width'] = img_w
            record['image_height'] = img_h
            record['file_name'] = f"images/{safe_fname}"
            
            # Ghi ảnh ra thư mục đích (Không dùng shutil.copy2 để không bị lây nhiễm quyền Read-Only)
            out_img_path = OUT_IMG_DIR / safe_fname
            if not out_img_path.exists():
                cv2.imwrite(str(out_img_path), img)
                
            out_meta_file.write(json.dumps(record, ensure_ascii=False) + '\n')
            process_and_add_to_datasets(f"images/{safe_fname}", img_w, img_h, regions)
            stats["original_images"] += 1
            
            # Lọc box chuẩn
            valid_bboxes, valid_region_indices = [], []
            for idx, r in enumerate(regions):
                if 'bbox' in r and len(r['bbox']) == 4:
                    coco_box = original_bbox_to_coco(r['bbox'], BBOX_FORMAT)
                    clipped = clamp_coco_bbox(coco_box, img_w, img_h)
                    if clipped[2] > 1 and clipped[3] > 1:
                        valid_bboxes.append(clipped)
                        valid_region_indices.append(idx)
                    else: stats["invalid_boxes_dropped"] += 1
            
            if not valid_bboxes: continue
                
            # -- BƯỚC 2: TÍNH ĐIỂM & AUGMENT --
            priority = compute_image_priority(regions)
            repeats = priority["repeats"]
            pipeline_name = priority["pipeline"]
            
            if repeats == 0: continue
                
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            transform = pipelines[pipeline_name]
            
            for i in range(repeats):
                try: 
                    augmented = transform(image=img_rgb, bboxes=valid_bboxes, region_idx=valid_region_indices)
                except Exception:
                    stats["failed_augments"] += 1
                    continue
                    
                aug_bboxes, aug_indices = augmented['bboxes'], augmented['region_idx']
                if not aug_bboxes: continue
                    
                stem = Path(safe_fname).stem
                ext = Path(safe_fname).suffix
                new_fname = f"{stem}_{pipeline_name}_aug{i+1:03d}{ext}"
                
                # -- FIX 3: Ghi ảnh an toàn, bắt lỗi trực tiếp --
                success = cv2.imwrite(str(OUT_IMG_DIR / new_fname), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
                if not success:
                    stats["failed_augments"] += 1
                    continue
                
                new_record = copy.deepcopy(record)
                new_record['file_name'] = f"images/{new_fname}"
                new_record['image_width'] = augmented['image'].shape[1]
                new_record['image_height'] = augmented['image'].shape[0]
                
                new_regions = []
                for new_bbox, orig_idx in zip(aug_bboxes, aug_indices):
                    orig_idx = int(orig_idx) 
                    region = copy.deepcopy(regions[orig_idx])
                    
                    final_bbox = clamp_coco_bbox(coco_bbox_to_original(new_bbox, BBOX_FORMAT), new_record['image_width'], new_record['image_height'])
                    region['bbox'] = [round(float(v), 2) for v in final_bbox]
                    new_regions.append(region)
                    stats["boxes_added"][region.get('type', 'unknown')] += 1
                    
                new_record['regions'] = new_regions
                out_meta_file.write(json.dumps(new_record, ensure_ascii=False) + '\n')
                process_and_add_to_datasets(f"images/{new_fname}", new_record['image_width'], new_record['image_height'], new_regions)
                stats["augmented_images"] += 1
                
                if pipeline_name == "aug_rare" or (random.random() < 0.1 and len(debug_samples) < 30):
                    if len(debug_samples) < 30: debug_samples.append((OUT_IMG_DIR / new_fname, new_record))
                    
    out_meta_file.close()
    
    print("\n💾 Đang xuất file COCO và Relations JSON...")
    with open(OUT_COCO_PATH, 'w', encoding='utf-8') as f: json.dump(coco_dataset, f, ensure_ascii=False, indent=2)
    with open(OUT_REL_PATH, 'w', encoding='utf-8') as f: json.dump(relations_dataset, f, ensure_ascii=False, indent=2)

    print("🔍 Đang tạo Debug Visualization...")
    for img_p, rec in debug_samples:
        dbg_img = cv2.imread(str(img_p))
        if dbg_img is None: continue
        for r in rec.get('regions', []):
            box = r['bbox']
            if BBOX_FORMAT == "coco": x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[0]+box[2]), int(box[1]+box[3])
            else: x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
            cv2.rectangle(dbg_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(dbg_img, r.get('type',''), (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.imwrite(str(DEBUG_DIR / img_p.name), dbg_img)

    print("\n📊 BÁO CÁO THỐNG KÊ TỔNG HỢP:")
    print(f" - Ảnh gốc: {stats['original_images']}")
    print(f" - Ảnh augment tạo thêm: {stats['augmented_images']}")
    print(f" - Box bị loại: {stats['invalid_boxes_dropped']}")
    
    print("\n📦 Phân bố Class (Trước -> Tăng thêm -> Final):")
    for c in sorted(set(stats['boxes_before'].keys()) | set(stats['boxes_added'].keys())):
        b, a = stats['boxes_before'].get(c, 0), stats['boxes_added'].get(c, 0)
        print(f"   + {c.ljust(15)}: {str(b).rjust(6)} -> +{str(a).rjust(6)} -> = {b+a}")

    print(f"\n🗜️ Đang nén thành {ZIP_OUT_PATH}.zip...")
    shutil.make_archive(ZIP_OUT_PATH, 'zip', OUT_ROOT)
    cleanup_working_except_zip(f"{ZIP_OUT_PATH}.zip")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


🚀 Bắt đầu gộp và xử lý bộ dữ liệu Merged_Rukopys_V1...

📂 Đang xử lý tập dữ liệu: [GOLD]


Processing gold: 100%|██████████| 1330/1330 [03:47<00:00,  5.84it/s]



📂 Đang xử lý tập dữ liệu: [SILVER]


Processing silver: 100%|██████████| 679/679 [02:28<00:00,  4.58it/s]



💾 Đang xuất file COCO và Relations JSON...
🔍 Đang tạo Debug Visualization...

📊 BÁO CÁO THỐNG KÊ TỔNG HỢP:
 - Ảnh gốc: 2009
 - Ảnh augment tạo thêm: 753
 - Box bị loại: 0

📦 Phân bố Class (Trước -> Tăng thêm -> Final):
   + annotation     :   1057 -> +   856 -> = 1913
   + formula        :   4040 -> +   899 -> = 4939
   + graph          :     98 -> +   166 -> = 264
   + handwritten    :  27651 -> +  5592 -> = 33243
   + image          :    556 -> +   983 -> = 1539
   + printed        :   3710 -> +  5858 -> = 9568
   + table          :    687 -> +   400 -> = 1087

🗜️ Đang nén thành /kaggle/working/Merged_Rukopys_V1.zip...
🧹 Đang dọn dẹp toàn bộ dữ liệu trung gian trong /kaggle/working...
✅ Cleanup hoàn tất. Chỉ giữ lại file Zip.
